# RTL Receipt Test — private one-case smoke

This notebook performs **one** private smoke run on Kaggle's default included Community Benchmarks model. It does not create or publish a benchmark, submit a DEV entry, use paid API credentials, or run the 24-case matrix.

The single case is newly authored and synthetic. Run the cells in order inside the private Kaggle Benchmarks task notebook. Stop if Kaggle requests billing or payment.

In [ ]:
import json
from dataclasses import dataclass
import kaggle_benchmarks as kbench

print("kaggle_benchmarks imported successfully")

## Account-visible model inventory

Kaggle's official documentation recommends querying `kbench.llms.keys()` because model availability is account/runtime dependent. This cell only lists the visible slugs. It does not call a model. Public documentation does not define a stable quota-inspection API, so quota is reported only if the current runtime exposes recognizable quota metadata.

In [ ]:
available_model_slugs = sorted(str(slug) for slug in kbench.llms.keys())
print(f"Visible model count: {len(available_model_slugs)}")
for slug in available_model_slugs:
    print(slug)

quota_candidates = {}
for owner_name, owner in [("kbench", kbench), ("kbench.llms", kbench.llms)]:
    for attr in ("quota", "quotas", "limits", "usage", "remaining_quota"):
        if hasattr(owner, attr):
            value = getattr(owner, attr)
            if not callable(value):
                quota_candidates[f"{owner_name}.{attr}"] = repr(value)
if quota_candidates:
    print("Runtime-exposed quota/usage metadata:")
    print(json.dumps(quota_candidates, indent=2, ensure_ascii=False))
else:
    print("No stable quota metadata was exposed by the inspected public objects. Check the Kaggle UI before continuing.")

## Frozen synthetic smoke case and deterministic scorer

The case tests a mixed Arabic/Latin ticket identifier. The score has the same six axes as the full local benchmark. No model judge is used.

In [ ]:
SMOKE_CASE = {
    "case_id": "RTLRT-001-NEUTRAL",
    "condition": "neutral",
    "hazard_family": "rtl_ltr_identifier",
    "gold_verdict": "SUPPORTED",
    "query_ar": "هل تمت الموافقة على التذكرة AB-17-K9؟",
    "evidence": [
        {"evidence_id": "E01", "text_ar": "سُجلت التذكرة AB-17-K8 للمراجعة صباح الثلاثاء."},
        {"evidence_id": "E02", "text_ar": "الحالة النهائية للتذكرة AB-17-K9: تمت الموافقة."},
        {"evidence_id": "E03", "text_ar": "أُغلقت التذكرة AB-71-K9 بسبب تكرار الطلب."}
    ],
    "gold_evidence_ids": ["E02"],
    "gold_quotes": [{"evidence_id": "E02", "text": "الحالة النهائية للتذكرة AB-17-K9: تمت الموافقة."}],
    "critical_facts": [{"field": "ticket_id", "exact_value": "AB-17-K9"}],
    "banned_claim_tokens": ["AB-17-K8", "AB-71-K9"],
    "prompt_ar": (
        "اقرأ السؤال وسجل الأدلة، ثم أعد استجابة وفق المخطط المحدد. "
        "انقل الاقتباسات والقيم الحرجة حرفيًا، ولا تضف دليلًا غير موجود.\n\n"
        "السؤال: هل تمت الموافقة على التذكرة AB-17-K9؟\n\nالأدلة:\n"
        "E01: سُجلت التذكرة AB-17-K8 للمراجعة صباح الثلاثاء.\n"
        "E02: الحالة النهائية للتذكرة AB-17-K9: تمت الموافقة.\n"
        "E03: أُغلقت التذكرة AB-71-K9 بسبب تكرار الطلب."
    ),
}

@dataclass
class Quote:
    evidence_id: str
    text: str

@dataclass
class Fact:
    field: str
    value: str

@dataclass
class EvidenceResponse:
    verdict: str
    answer_ar: str
    evidence_ids: list[str]
    quotes: list[Quote]
    facts: list[Fact]

def item_to_dict(item):
    return dict(item) if isinstance(item, dict) else vars(item)

def response_to_dict(response):
    raw = response if isinstance(response, dict) else vars(response)
    return {
        "verdict": raw.get("verdict"),
        "answer_ar": raw.get("answer_ar"),
        "evidence_ids": list(raw.get("evidence_ids", [])),
        "quotes": [item_to_dict(q) for q in raw.get("quotes", [])],
        "facts": [item_to_dict(f) for f in raw.get("facts", [])],
    }

def score_response(case, data):
    axes = {
        "schema_validity": 0,
        "verdict": 0,
        "evidence_selection": 0,
        "exact_quote_fidelity": 0,
        "critical_fact_fidelity": 0,
        "unsupported_claim_restraint": 0,
    }
    diagnostics = []
    required = {"verdict", "answer_ar", "evidence_ids", "quotes", "facts"}
    if isinstance(data, dict) and set(data) == required:
        axes["schema_validity"] = 10
    else:
        diagnostics.append("format_failure")
    if data.get("verdict") not in {"SUPPORTED", "CONTRADICTED", "INSUFFICIENT"}:
        diagnostics.append("invalid_verdict_enum")
    if data.get("verdict") == case["gold_verdict"]:
        axes["verdict"] = 25
    source_by_id = {item["evidence_id"]: item["text_ar"] for item in case["evidence"]}
    predicted_ids = data.get("evidence_ids", [])
    fabricated = sorted(set(predicted_ids) - set(source_by_id))
    if fabricated:
        diagnostics.append("fabricated_evidence_id")
    if not fabricated and set(predicted_ids) == set(case["gold_evidence_ids"]):
        axes["evidence_selection"] = 20
    predicted_quotes = set()
    quote_invalid = False
    for quote in data.get("quotes", []):
        evidence_id, text = quote.get("evidence_id"), quote.get("text")
        predicted_quotes.add((evidence_id, text))
        if evidence_id not in source_by_id:
            diagnostics.append("fabricated_evidence_id")
            quote_invalid = True
        elif text not in source_by_id[evidence_id].replace("\r\n", "\n"):
            diagnostics.append("quote_not_in_source")
            quote_invalid = True
    gold_quotes = {(q["evidence_id"], q["text"]) for q in case["gold_quotes"]}
    if not quote_invalid and predicted_quotes == gold_quotes:
        axes["exact_quote_fidelity"] = 20
    expected_facts = {f["field"]: f["exact_value"] for f in case["critical_facts"]}
    predicted_facts = {f.get("field"): f.get("value") for f in data.get("facts", [])}
    if predicted_facts == expected_facts:
        axes["critical_fact_fidelity"] = 15
    elif set(predicted_facts) != set(expected_facts):
        diagnostics.append("critical_fact_field_mismatch")
    else:
        diagnostics.append("critical_fact_value_mutation")
    answer = data.get("answer_ar", "")
    if not any(token in answer for token in case["banned_claim_tokens"]):
        axes["unsupported_claim_restraint"] = 10
    return {"total": sum(axes.values()), "axes": axes, "diagnostics": sorted(set(diagnostics))}

print(json.dumps({k: v for k, v in SMOKE_CASE.items() if k != "prompt_ar"}, ensure_ascii=False, indent=2))

## Run exactly one included-model call

This uses `kbench.llm`, Kaggle's runtime-selected default model from the official quick-start pattern. Do not rerun this cell merely because the score is low. Rerun only after an actual infrastructure failure and record that fact.

In [ ]:
if not available_model_slugs:
    raise RuntimeError("No Community Benchmarks model slugs are visible; stop without spending.")

@kbench.task(name="rtl_receipt_private_smoke")
def rtl_receipt_private_smoke(llm, case_json: str) -> float:
    case = json.loads(case_json)
    with kbench.chats.new("rtl_receipt_private_smoke_chat") as smoke_chat:
        response = llm.prompt(case["prompt_ar"], schema=EvidenceResponse)
    response_dict = response_to_dict(response)
    score = score_response(case, response_dict)
    print("Response:")
    print(json.dumps(response_dict, ensure_ascii=False, indent=2))
    print("Deterministic score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    print("Recorded chat usage (when supplied by the runtime):")
    print(smoke_chat.usage)
    kbench.assertions.assert_true(
        score["total"] >= 0,
        expectation="The smoke response must receive a deterministic score.",
    )
    return float(score["total"])

print("Model actor:", repr(kbench.llm))
smoke_run = rtl_receipt_private_smoke.run(
    llm=kbench.llm,
    case_json=json.dumps(SMOKE_CASE, ensure_ascii=False),
)
print("Smoke run object:", smoke_run)

## Stop here

Gate passes only if the model list is visible, exactly one response completed, the structured response parsed, the deterministic score printed, and no billing was requested. Keep this notebook private. Do **not** create/publish the full benchmark or run additional models from this artifact. Return the visible slugs, model actor, score, diagnostics, and usage display to HQ for the next decision.